# Sobre o Que Eles Falam? Análise de Conteúdo: Trump vs AOC
## Examinando Linguagem, Tópicos e Identidade Digital

Neste notebook, analisamos **o que** Trump e AOC realmente dizem em seus tweets e **como** dizem. Esta é uma parte fundamental da **netnografia** -- o estudo de comunidades online e culturas. Ao analisar as palavras que os políticos escolhem, as hashtags que usam e os tópicos em que focam, podemos entender como eles constroem sua imagem pública e se conectam com suas audiências nas redes sociais.

**Por que a linguagem importa na política?** Cada palavra que um político digita é uma escolha. Usar MAIÚSCULAS sinaliza urgência ou raiva. Escolher certas hashtags sinaliza pertencimento a um movimento. Mencionar outros políticos constrói alianças ou compra brigas. Ao estudar esses padrões sistematicamente, vamos além de impressões intuitivas e entramos em uma análise baseada em evidências da comunicação política.

**O que você aprenderá neste notebook:**
1. Como Trump e AOC diferem no estilo de escrita (tamanho, pontuação, capitalização)
2. Quais palavras cada político mais usa -- e quais palavras são únicas de cada um
3. Como as hashtags servem como marcadores de identidade e gritos de guerra
4. Quem cada político menciona e o que isso revela sobre suas redes
5. Quais tópicos dominam o feed de cada político
6. Um "mapa" visual de todos os tweets mostrando como a IA entende o significado de cada mensagem

In [ ]:
# --- Configuração: imports e configuração ---

import sys
sys.path.insert(0, '../..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

from utils.data_loader import load_tweets
from utils.plot_helpers import (
    setup_style, get_colors, get_user_label,
    comparative_bar, comparative_hist, format_large_numbers,
    set_language
)
from utils.text_helpers import (
    extract_hashtags, extract_mentions, clean_text,
    get_word_frequencies, get_stopwords
)
from config import (
    FIGURE_SIZE, FIGURE_SIZE_SMALL, FIGURE_SIZE_LARGE,
    RANDOM_SEED, TARGET_USERS
)

# Aplicar estilo consistente
setup_style()
set_language('pt-br')
colors = get_colors()
np.random.seed(RANDOM_SEED)

# Configurações de exibição
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

print('Configuração concluída.')

In [ ]:
# --- Carregar dados ---

df = load_tweets()

print(f"Formato: {df.shape}")
print(f"Usuários: {df['user'].value_counts().to_dict()}")
print(f"Intervalo de datas: {df['created_at'].min()} até {df['created_at'].max()}")

## 1. Características do Texto

O tamanho, capitalização, pontuação e formatação dos tweets revelam diferenças fundamentais em como cada político aborda a comunicação digital. Essas características servem como indicadores mensuráveis de tom, urgência e intensidade emocional.

In [ ]:
# --- Comparação de características do texto: grade 2x3 ---

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# --- Linha 1: Histogramas de distribuição ---

# 1. Distribuições de tamanho do texto
ax = axes[0, 0]
for user in ['trump', 'aoc']:
    user_data = df[df['user'] == user]['text_length'].dropna()
    ax.hist(user_data, bins=20, alpha=0.55, color=colors[user],
            edgecolor='white', linewidth=0.5, label=get_user_label(user))
    mean_val = user_data.mean()
    ax.axvline(mean_val, color=colors[user], linestyle='--', linewidth=1.5, alpha=0.8)
ax.set_title('Distribuição do Tamanho do Texto', fontweight='bold')
ax.set_xlabel('Caracteres')
ax.set_ylabel('Frequência')
ax.legend(fontsize=8)

# 2. Distribuições de contagem de palavras
ax = axes[0, 1]
for user in ['trump', 'aoc']:
    user_data = df[df['user'] == user]['word_count'].dropna()
    ax.hist(user_data, bins=20, alpha=0.55, color=colors[user],
            edgecolor='white', linewidth=0.5, label=get_user_label(user))
    mean_val = user_data.mean()
    ax.axvline(mean_val, color=colors[user], linestyle='--', linewidth=1.5, alpha=0.8)
ax.set_title('Distribuição da Contagem de Palavras', fontweight='bold')
ax.set_xlabel('Palavras')
ax.set_ylabel('Frequência')
ax.legend(fontsize=8)

# 3. Distribuições da razão de MAIÚSCULAS
ax = axes[0, 2]
for user in ['trump', 'aoc']:
    user_data = df[df['user'] == user]['caps_ratio'].dropna()
    ax.hist(user_data, bins=20, alpha=0.55, color=colors[user],
            edgecolor='white', linewidth=0.5, label=get_user_label(user))
    mean_val = user_data.mean()
    ax.axvline(mean_val, color=colors[user], linestyle='--', linewidth=1.5, alpha=0.8)
ax.set_title('Distribuição da Razão de MAIÚSCULAS', fontweight='bold')
ax.set_xlabel('Razão de Maiúsculas')
ax.set_ylabel('Frequência')
ax.legend(fontsize=8)

# --- Linha 2: Gráficos de barras ---

# 4. Frequência de exclamações (média por tweet)
ax = axes[1, 0]
users = ['trump', 'aoc']
exc_means = [df[df['user'] == u]['exclamation_count'].mean() for u in users]
user_labels = [get_user_label(u) for u in users]
bars = ax.bar(user_labels, exc_means, color=[colors[u] for u in users],
              width=0.5, edgecolor='white')
for bar, val in zip(bars, exc_means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f'{val:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('Média de Exclamações por Tweet', fontweight='bold')
ax.set_ylabel('Quantidade')
ax.set_ylim(0, max(exc_means) * 1.25 if max(exc_means) > 0 else 1)
ax.tick_params(axis='x', rotation=15)

# 5. Frequência de interrogações (média por tweet)
ax = axes[1, 1]
q_means = [df[df['user'] == u]['question_count'].mean() for u in users]
bars = ax.bar(user_labels, q_means, color=[colors[u] for u in users],
              width=0.5, edgecolor='white')
for bar, val in zip(bars, q_means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f'{val:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('Média de Interrogações por Tweet', fontweight='bold')
ax.set_ylabel('Quantidade')
ax.set_ylim(0, max(q_means) * 1.25 if max(q_means) > 0 else 1)
ax.tick_params(axis='x', rotation=15)

# 6. Porcentagem de uso de emoji
ax = axes[1, 2]
emoji_pcts = [df[df['user'] == u]['has_emoji'].mean() * 100 for u in users]
bars = ax.bar(user_labels, emoji_pcts, color=[colors[u] for u in users],
              width=0.5, edgecolor='white')
for bar, val in zip(bars, emoji_pcts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('Tweets Contendo Emoji (%)', fontweight='bold')
ax.set_ylabel('Porcentagem')
ax.set_ylim(0, max(emoji_pcts) * 1.25 if max(emoji_pcts) > 0 else 10)
ax.tick_params(axis='x', rotation=15)

fig.suptitle('Comparação de Características do Texto', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Tabela de estatísticas resumidas
print("\nResumo de Características do Texto:")
text_stats = df.groupby('user').agg({
    'text_length': ['mean', 'median', 'std'],
    'word_count': ['mean', 'median', 'std'],
    'caps_ratio': ['mean', 'median'],
    'exclamation_count': ['mean', 'sum'],
    'question_count': ['mean', 'sum'],
}).round(2)
print(text_stats.to_string())

In [ ]:
# --- Análise aprofundada da razão de maiúsculas ---

fig, ax = plt.subplots(figsize=FIGURE_SIZE_SMALL)

users = ['trump', 'aoc']
caps_means = [df[df['user'] == u]['caps_ratio'].mean() for u in users]
user_labels = [get_user_label(u) for u in users]

bars = ax.bar(user_labels, caps_means, color=[colors[u] for u in users],
              width=0.5, edgecolor='white')
for bar, val in zip(bars, caps_means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f'{val:.3f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_title('Razão Média de MAIÚSCULAS por Usuário', fontweight='bold', fontsize=14)
ax.set_ylabel('Razão de Letras Maiúsculas')
ax.set_ylim(0, max(caps_means) * 1.3 if max(caps_means) > 0 else 0.5)
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

# Mostrar exemplos de tweets com maior caps_ratio
print("\n" + "=" * 80)
print("TWEETS COM MAIOR RAZÃO DE MAIÚSCULAS")
print("=" * 80)

for user in ['trump', 'aoc']:
    user_df = df[df['user'] == user].nlargest(3, 'caps_ratio')
    label = get_user_label(user)
    print(f"\n--- {label} ---")
    for rank, (_, row) in enumerate(user_df.iterrows(), 1):
        text_preview = row['text'][:120] + '...' if len(str(row['text'])) > 120 else row['text']
        print(f"  {rank}. [razão_maiúsc={row['caps_ratio']:.3f}] {text_preview}")

## 2. Análise de Frequência de Palavras

As palavras mais frequentemente usadas revelam as prioridades temáticas de cada político. Escolhas vocabulares não são neutras -- elas enquadram questões, sinalizam alinhamento ideológico e constroem identidade de grupo. Ao examinar distribuições de frequência de palavras, podemos identificar os blocos fundamentais da persona digital de cada político.

In [ ]:
# --- Top 30 palavras para cada usuário ---

fig, axes = plt.subplots(1, 2, figsize=FIGURE_SIZE_LARGE)

for idx, user in enumerate(['trump', 'aoc']):
    ax = axes[idx]
    user_texts = df[df['user'] == user]['text'].dropna().tolist()
    word_freqs = get_word_frequencies(user_texts, top_n=30)

    if word_freqs:
        words, counts = zip(*reversed(word_freqs))  # invertido para barra horizontal
        y_pos = np.arange(len(words))

        ax.barh(y_pos, counts, color=colors[user], alpha=0.85,
                edgecolor='white', linewidth=0.5)
        ax.set_yticks(y_pos)
        ax.set_yticklabels(words, fontsize=9)

        # Adicionar rótulos de contagem
        for i, count in enumerate(counts):
            ax.text(count + max(counts) * 0.01, i, str(count),
                    va='center', fontsize=8, fontweight='bold')

    ax.set_title(f'{get_user_label(user)}', fontweight='bold', fontsize=13)
    ax.set_xlabel('Frequência')
    ax.set_xlim(0, max(counts) * 1.12 if word_freqs else 10)

fig.suptitle('Top 30 Palavras Mais Frequentes (Stopwords Removidas)',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Vocabulário exclusivo: palavras frequentes para um usuário mas não para o outro ---

# Obter frequências de palavras para cada usuário (conjunto maior para comparação)
trump_texts = df[df['user'] == 'trump']['text'].dropna().tolist()
aoc_texts = df[df['user'] == 'aoc']['text'].dropna().tolist()

trump_freqs = dict(get_word_frequencies(trump_texts, top_n=200))
aoc_freqs = dict(get_word_frequencies(aoc_texts, top_n=200))

# Encontrar palavras distintivas: alta frequência em um, baixa/ausente no outro
# Pontuação = frequência no usuário / (frequência no outro + 1)
trump_distinctive = []
for word, count in trump_freqs.items():
    other_count = aoc_freqs.get(word, 0)
    score = count / (other_count + 1)
    if score >= 2 and count >= 3:  # pelo menos 2x mais frequente e aparece 3+ vezes
        trump_distinctive.append((word, count, score))

aoc_distinctive = []
for word, count in aoc_freqs.items():
    other_count = trump_freqs.get(word, 0)
    score = count / (other_count + 1)
    if score >= 2 and count >= 3:
        aoc_distinctive.append((word, count, score))

# Ordenar por pontuação de distinção e pegar top 15
trump_distinctive.sort(key=lambda x: x[2], reverse=True)
aoc_distinctive.sort(key=lambda x: x[2], reverse=True)
trump_top15 = trump_distinctive[:15]
aoc_top15 = aoc_distinctive[:15]

# Plotar
fig, axes = plt.subplots(1, 2, figsize=FIGURE_SIZE_LARGE)

for idx, (user, data) in enumerate([('trump', trump_top15), ('aoc', aoc_top15)]):
    ax = axes[idx]
    if data:
        words = [d[0] for d in reversed(data)]
        counts = [d[1] for d in reversed(data)]
        y_pos = np.arange(len(words))

        ax.barh(y_pos, counts, color=colors[user], alpha=0.85,
                edgecolor='white', linewidth=0.5)
        ax.set_yticks(y_pos)
        ax.set_yticklabels(words, fontsize=9)

        for i, count in enumerate(counts):
            ax.text(count + max(counts) * 0.01, i, str(count),
                    va='center', fontsize=8, fontweight='bold')

        ax.set_xlim(0, max(counts) * 1.15)
    else:
        ax.text(0.5, 0.5, 'Nenhuma palavra distintiva encontrada',
                ha='center', va='center', transform=ax.transAxes, fontsize=12)

    ax.set_title(f'{get_user_label(user)} -- Palavras Distintivas', fontweight='bold', fontsize=12)
    ax.set_xlabel('Frequência')

fig.suptitle('Top 15 Palavras Distintivas por Usuário\n(Palavras desproporcionalmente usadas por um usuário)',
             fontsize=15, fontweight='bold', y=1.04)
plt.tight_layout()
plt.show()

# Imprimir detalhes
print("\nDetalhes do vocabulário distintivo:")
for user, data in [('trump', trump_top15), ('aoc', aoc_top15)]:
    label = get_user_label(user)
    print(f"\n--- {label} ---")
    for word, count, score in data:
        print(f"  '{word}': usado {count}x (pontuação de distinção: {score:.1f})")

## 3. Análise de Hashtags

Hashtags funcionam simultaneamente como rótulos temáticos, sinais ideológicos e ferramentas de mobilização. Seu uso estratégico revela como cada político se posiciona dentro de conversas online mais amplas e constrói afinidade com movimentos ou comunidades específicas.

In [ ]:
# --- Visão geral do uso de hashtags ---

fig, axes = plt.subplots(1, 2, figsize=FIGURE_SIZE)

users = ['trump', 'aoc']
user_labels = [get_user_label(u) for u in users]

# 1. Porcentagem de tweets contendo pelo menos uma hashtag
ax = axes[0]
hashtag_pcts = []
for u in users:
    user_data = df[df['user'] == u]
    pct = (user_data['hashtag_count'] > 0).mean() * 100
    hashtag_pcts.append(pct)

bars = ax.bar(user_labels, hashtag_pcts, color=[colors[u] for u in users],
              width=0.5, edgecolor='white')
for bar, val in zip(bars, hashtag_pcts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f'{val:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_title('Tweets com Hashtags (%)', fontweight='bold', fontsize=13)
ax.set_ylabel('Porcentagem')
ax.set_ylim(0, max(hashtag_pcts) * 1.25 if max(hashtag_pcts) > 0 else 10)
ax.tick_params(axis='x', rotation=15)

# 2. Média de hashtags por tweet
ax = axes[1]
hashtag_means = [df[df['user'] == u]['hashtag_count'].mean() for u in users]
bars = ax.bar(user_labels, hashtag_means, color=[colors[u] for u in users],
              width=0.5, edgecolor='white')
for bar, val in zip(bars, hashtag_means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f'{val:.2f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_title('Média de Hashtags por Tweet', fontweight='bold', fontsize=13)
ax.set_ylabel('Quantidade')
ax.set_ylim(0, max(hashtag_means) * 1.25 if max(hashtag_means) > 0 else 1)
ax.tick_params(axis='x', rotation=15)

fig.suptitle('Visão Geral do Uso de Hashtags', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Top 15 hashtags por usuário ---

fig, axes = plt.subplots(1, 2, figsize=FIGURE_SIZE_LARGE)

for idx, user in enumerate(['trump', 'aoc']):
    ax = axes[idx]
    user_df = df[df['user'] == user]

    # Extrair todas as hashtags da coluna hashtags_extracted
    all_hashtags = []
    for tags_str in user_df['hashtags_extracted'].dropna():
        if isinstance(tags_str, str):
            try:
                import ast
                tags = ast.literal_eval(tags_str)
                if isinstance(tags, list):
                    all_hashtags.extend([t.lower() for t in tags])
                else:
                    all_hashtags.append(str(tags).lower())
            except (ValueError, SyntaxError):
                for tag in tags_str.replace('[', '').replace(']', '').replace("'", '').split(','):
                    tag = tag.strip().lower()
                    if tag:
                        all_hashtags.append(tag)

    # Contar e obter top 15
    hashtag_counts = Counter(all_hashtags).most_common(15)

    if hashtag_counts:
        tags, counts = zip(*reversed(hashtag_counts))
        y_pos = np.arange(len(tags))

        ax.barh(y_pos, counts, color=colors[user], alpha=0.85,
                edgecolor='white', linewidth=0.5)
        ax.set_yticks(y_pos)
        ax.set_yticklabels([f'#{t}' for t in tags], fontsize=9)

        for i, count in enumerate(counts):
            ax.text(count + max(counts) * 0.01, i, str(count),
                    va='center', fontsize=8, fontweight='bold')

        ax.set_xlim(0, max(counts) * 1.15)
    else:
        ax.text(0.5, 0.5, 'Nenhuma hashtag encontrada',
                ha='center', va='center', transform=ax.transAxes, fontsize=12)

    ax.set_title(f'{get_user_label(user)}', fontweight='bold', fontsize=13)
    ax.set_xlabel('Frequência')

fig.suptitle('Top 15 Hashtags por Usuário', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Hashtags compartilhadas vs exclusivas ---

import ast

def extract_all_hashtags(user_df):
    """Extrair todas as hashtags únicas (em minúsculas) dos tweets de um usuário."""
    all_tags = set()
    for tags_str in user_df['hashtags_extracted'].dropna():
        if isinstance(tags_str, str):
            try:
                tags = ast.literal_eval(tags_str)
                if isinstance(tags, list):
                    all_tags.update(t.lower() for t in tags)
                else:
                    all_tags.add(str(tags).lower())
            except (ValueError, SyntaxError):
                for tag in tags_str.replace('[', '').replace(']', '').replace("'", '').split(','):
                    tag = tag.strip().lower()
                    if tag:
                        all_tags.add(tag)
    return all_tags

trump_hashtags = extract_all_hashtags(df[df['user'] == 'trump'])
aoc_hashtags = extract_all_hashtags(df[df['user'] == 'aoc'])

shared = trump_hashtags & aoc_hashtags
trump_only = trump_hashtags - aoc_hashtags
aoc_only = aoc_hashtags - trump_hashtags

# Exibir resultados
print("=" * 70)
print("ANÁLISE DO CONJUNTO DE HASHTAGS")
print("=" * 70)
print(f"\nHashtags únicas do Trump: {len(trump_hashtags)}")
print(f"Hashtags únicas da AOC:   {len(aoc_hashtags)}")
print(f"\nHashtags compartilhadas ({len(shared)}):")
if shared:
    for tag in sorted(shared):
        print(f"  #{tag}")
else:
    print("  Nenhuma -- sem sobreposição de hashtags entre os usuários.")

print(f"\nHashtags exclusivas do Trump ({len(trump_only)}):")
for tag in sorted(list(trump_only)[:20]):
    print(f"  #{tag}")
if len(trump_only) > 20:
    print(f"  ... e mais {len(trump_only) - 20}")

print(f"\nHashtags exclusivas da AOC ({len(aoc_only)}):")
for tag in sorted(list(aoc_only)[:20]):
    print(f"  #{tag}")
if len(aoc_only) > 20:
    print(f"  ... e mais {len(aoc_only) - 20}")

# Gráfico de barras mostrando as contagens
fig, ax = plt.subplots(figsize=FIGURE_SIZE_SMALL)
categories = ['Compartilhadas', 'Exclusivas Trump', 'Exclusivas AOC']
counts = [len(shared), len(trump_only), len(aoc_only)]
bar_colors = ['#6C757D', colors['trump'], colors['aoc']]

bars = ax.bar(categories, counts, color=bar_colors, width=0.5, edgecolor='white')
for bar, val in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            str(val), ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_title('Hashtags Compartilhadas vs Exclusivas', fontweight='bold', fontsize=14)
ax.set_ylabel('Número de Hashtags Únicas')
ax.set_ylim(0, max(counts) * 1.2 if max(counts) > 0 else 5)
plt.tight_layout()
plt.show()

## 4. URLs e Compartilhamento de Links

Quando um político compartilha um link, está curando o ambiente informacional de sua audiência. A frequência e os alvos do compartilhamento de URLs indicam se um político se posiciona como criador primário de conteúdo ou como curador de fontes externas.

In [ ]:
# --- Padrões de compartilhamento de links ---

fig, axes = plt.subplots(1, 2, figsize=FIGURE_SIZE)

users = ['trump', 'aoc']
user_labels = [get_user_label(u) for u in users]

# 1. Porcentagem de tweets com URLs
ax = axes[0]
url_pcts = [(df[df['user'] == u]['url_count'] > 0).mean() * 100 for u in users]
bars = ax.bar(user_labels, url_pcts, color=[colors[u] for u in users],
              width=0.5, edgecolor='white')
for bar, val in zip(bars, url_pcts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f'{val:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_title('Tweets Contendo URLs (%)', fontweight='bold', fontsize=13)
ax.set_ylabel('Porcentagem')
ax.set_ylim(0, max(url_pcts) * 1.25 if max(url_pcts) > 0 else 10)
ax.tick_params(axis='x', rotation=15)

# 2. Média de URLs por tweet
ax = axes[1]
url_means = [df[df['user'] == u]['url_count'].mean() for u in users]
bars = ax.bar(user_labels, url_means, color=[colors[u] for u in users],
              width=0.5, edgecolor='white')
for bar, val in zip(bars, url_means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f'{val:.2f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_title('Média de URLs por Tweet', fontweight='bold', fontsize=13)
ax.set_ylabel('Quantidade')
ax.set_ylim(0, max(url_means) * 1.25 if max(url_means) > 0 else 1)
ax.tick_params(axis='x', rotation=15)

fig.suptitle('Padrões de Compartilhamento de URL / Links', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Resumo
print("\nResumo de Compartilhamento de URLs:")
for u in users:
    user_data = df[df['user'] == u]
    total = len(user_data)
    with_url = (user_data['url_count'] > 0).sum()
    mean_urls = user_data['url_count'].mean()
    max_urls = user_data['url_count'].max()
    print(f"  {get_user_label(u)}: {with_url}/{total} tweets com URLs "
          f"({with_url/total*100:.1f}%), média={mean_urls:.2f}, máx={max_urls}")

## 5. Padrões de Menções

Menções são uma forma de endereçamento digital -- invocam outros atores, constroem alianças ou miram adversários. O padrão de menções revela o cenário político percebido por cada político e sua estratégia para engajar com outras figuras públicas.

In [ ]:
# --- Top contas mencionadas por usuário ---

fig, axes = plt.subplots(1, 2, figsize=FIGURE_SIZE_LARGE)

for idx, user in enumerate(['trump', 'aoc']):
    ax = axes[idx]
    user_texts = df[df['user'] == user]['text'].dropna().tolist()

    # Extrair todas as menções
    all_mentions = []
    for text in user_texts:
        mentions = extract_mentions(text)
        all_mentions.extend([m.lower() for m in mentions])

    # Contar e obter top 15
    mention_counts = Counter(all_mentions).most_common(15)

    if mention_counts:
        accounts, counts = zip(*reversed(mention_counts))
        y_pos = np.arange(len(accounts))

        ax.barh(y_pos, counts, color=colors[user], alpha=0.85,
                edgecolor='white', linewidth=0.5)
        ax.set_yticks(y_pos)
        ax.set_yticklabels([f'@{a}' for a in accounts], fontsize=9)

        for i, count in enumerate(counts):
            ax.text(count + max(counts) * 0.01, i, str(count),
                    va='center', fontsize=8, fontweight='bold')

        ax.set_xlim(0, max(counts) * 1.15)
    else:
        ax.text(0.5, 0.5, 'Nenhuma menção encontrada',
                ha='center', va='center', transform=ax.transAxes, fontsize=12)

    ax.set_title(f'{get_user_label(user)}', fontweight='bold', fontsize=13)
    ax.set_xlabel('Frequência')

fig.suptitle('Top 15 Contas Mencionadas por Usuário',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Análise de menções cruzadas: Trump menciona AOC ou vice-versa? ---

print("=" * 70)
print("ANÁLISE DE MENÇÕES CRUZADAS")
print("=" * 70)

# Trump mencionando AOC
trump_texts = df[df['user'] == 'trump']['text'].dropna()
trump_mentions_aoc = trump_texts.str.lower().str.contains('aoc|ocasio|alexandria', regex=True).sum()
print(f"\nTrump mencionando AOC (texto contém 'aoc', 'ocasio' ou 'alexandria'):")
print(f"  Quantidade: {trump_mentions_aoc} de {len(trump_texts)} tweets")
if trump_mentions_aoc > 0:
    aoc_mention_tweets = df[(df['user'] == 'trump') & 
                            df['text'].str.lower().str.contains('aoc|ocasio|alexandria', regex=True, na=False)]
    print("  Tweets:")
    for _, row in aoc_mention_tweets.iterrows():
        preview = row['text'][:120] + '...' if len(str(row['text'])) > 120 else row['text']
        print(f"    - {preview}")

# AOC mencionando Trump
aoc_texts = df[df['user'] == 'aoc']['text'].dropna()
aoc_mentions_trump = aoc_texts.str.lower().str.contains('trump|donald', regex=True).sum()
print(f"\nAOC mencionando Trump (texto contém 'trump' ou 'donald'):")
print(f"  Quantidade: {aoc_mentions_trump} de {len(aoc_texts)} tweets")
if aoc_mentions_trump > 0:
    trump_mention_tweets = df[(df['user'] == 'aoc') & 
                              df['text'].str.lower().str.contains('trump|donald', regex=True, na=False)]
    print("  Tweets:")
    for _, row in trump_mention_tweets.head(10).iterrows():
        preview = row['text'][:120] + '...' if len(str(row['text'])) > 120 else row['text']
        print(f"    - {preview}")
    if len(trump_mention_tweets) > 10:
        print(f"    ... e mais {len(trump_mention_tweets) - 10}")

print(f"\n--- Resumo ---")
print(f"Referências de Trump a AOC: {trump_mentions_aoc} ({trump_mentions_aoc/len(trump_texts)*100:.1f}% dos tweets)")
print(f"Referências de AOC a Trump: {aoc_mentions_trump} ({aoc_mentions_trump/len(aoc_texts)*100:.1f}% dos tweets)")

## 6. Classificação de Tópicos

Embora não substitua uma leitura qualitativa aprofundada, a detecção de tópicos por palavras-chave fornece uma primeira aproximação escalável da agenda de temas de cada político. A distribuição de tópicos revela prioridades estratégicas e o enquadramento da identidade política.

In [ ]:
# --- Classificação de tópicos usando correspondência de palavras-chave ---

# Definir listas de palavras-chave por tópico
topic_keywords = {
    'Economia/Empregos': ['economy', 'jobs', 'tax', 'inflation', 'trade', 'tariff',
                     'business', 'gdp', 'debt', 'market'],
    'Imigração': ['border', 'immigration', 'immigrant', 'deportation', 'ice',
                    'asylum', 'migrant'],
    'Saúde': ['health', 'healthcare', 'medicare', 'medicaid', 'insurance',
                   'hospital'],
    'Política Externa': ['china', 'russia', 'ukraine', 'nato', 'war', 'military',
                       'iran', 'israel'],
    'Governo/Política': ['congress', 'senate', 'house', 'vote', 'election',
                            'democrat', 'republican', 'gop'],
    'Questões Sociais': ['education', 'climate', 'housing', 'gun', 'abortion',
                      'rights', 'equality', 'justice'],
    'Pessoal/Ataque': ['fake', 'corrupt', 'crooked', 'radical', 'socialist',
                        'fascist', 'liar'],
    'Autopromoção': ['maga', 'trump', 'great', 'best', 'winning', 'deal'],
}

# Classificar cada tweet (pode pertencer a múltiplos tópicos)
for topic, keywords in topic_keywords.items():
    col_name = f'topic_{topic.lower().replace("/", "_").replace(" ", "_")}'
    pattern = '|'.join(keywords)
    df[col_name] = df['text'].str.lower().str.contains(pattern, regex=True, na=False)

# Calcular distribuição de tópicos por usuário
topic_cols = [c for c in df.columns if c.startswith('topic_')]
topic_display_names = list(topic_keywords.keys())

topic_data = {}
for user in ['trump', 'aoc']:
    user_df = df[df['user'] == user]
    total = len(user_df)
    pcts = []
    for col in topic_cols:
        pct = user_df[col].sum() / total * 100
        pcts.append(pct)
    topic_data[user] = pcts

# Gráfico de barras agrupadas
fig, ax = plt.subplots(figsize=FIGURE_SIZE_LARGE)

x = np.arange(len(topic_display_names))
bar_width = 0.35

for i, user in enumerate(['trump', 'aoc']):
    offset = -bar_width / 2 + i * bar_width
    bars = ax.bar(x + offset, topic_data[user], bar_width,
                  label=get_user_label(user), color=colors[user],
                  alpha=0.85, edgecolor='white', linewidth=0.5)

    # Adicionar rótulos de valor
    for bar, val in zip(bars, topic_data[user]):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                    f'{val:.0f}%', ha='center', va='bottom',
                    fontsize=8, fontweight='bold')

ax.set_title('Distribuição de Tópicos por Usuário (Classificação por Palavras-Chave)',
             fontweight='bold', fontsize=14)
ax.set_xlabel('Tópico')
ax.set_ylabel('% de Tweets Correspondentes ao Tópico')
ax.set_xticks(x)
ax.set_xticklabels(topic_display_names, rotation=30, ha='right', fontsize=10)
ax.legend(fontsize=10)
ax.set_ylim(0, max(max(topic_data['trump']), max(topic_data['aoc'])) * 1.2 if 
            max(max(topic_data['trump']), max(topic_data['aoc'])) > 0 else 10)
plt.tight_layout()
plt.show()

# Imprimir tabela detalhada
print("\nResumo da Classificação de Tópicos (% de tweets correspondentes a cada tópico):")
print("-" * 60)
print(f"{'Tópico':<22} {'Trump':>10} {'AOC':>10} {'Diferença':>12}")
print("-" * 60)
for topic, trump_pct, aoc_pct in zip(topic_display_names,
                                      topic_data['trump'],
                                      topic_data['aoc']):
    diff = trump_pct - aoc_pct
    arrow = '>' if diff > 0 else '<' if diff < 0 else '='
    print(f"{topic:<22} {trump_pct:>9.1f}% {aoc_pct:>9.1f}% {diff:>+10.1f}pp {arrow}")

# Nota sobre cobertura
for user in ['trump', 'aoc']:
    user_df = df[df['user'] == user]
    any_topic = user_df[topic_cols].any(axis=1).sum()
    print(f"\n{get_user_label(user)}: {any_topic}/{len(user_df)} tweets corresponderam a pelo menos um tópico ({any_topic/len(user_df)*100:.1f}%)")

## 7. Síntese da Identidade Digital

Reunindo os padrões linguísticos, temáticos e estruturais observados acima, esta seção constrói um perfil comparativo da identidade digital de cada político -- a persona que projetam por meio do conteúdo e composição de seus tweets.

In [ ]:
# --- Tabela comparativa de estratégias ---

def get_top_topic(user):
    """Obter o tópico mais comum para um usuário."""
    user_df = df[df['user'] == user]
    topic_pcts = {}
    for col, name in zip(topic_cols, topic_display_names):
        topic_pcts[name] = user_df[col].sum()
    if topic_pcts:
        return max(topic_pcts, key=topic_pcts.get)
    return 'N/A'

def get_hashtag_strategy(user):
    """Descrever o padrão de uso de hashtags."""
    user_df = df[df['user'] == user]
    pct_with = (user_df['hashtag_count'] > 0).mean() * 100
    mean_count = user_df['hashtag_count'].mean()
    if pct_with < 10:
        return f'Mínimo ({pct_with:.0f}% dos tweets, {mean_count:.1f} média)'
    elif pct_with < 40:
        return f'Moderado ({pct_with:.0f}% dos tweets, {mean_count:.1f} média)'
    else:
        return f'Intenso ({pct_with:.0f}% dos tweets, {mean_count:.1f} média)'

def get_tone_descriptor(user):
    """Descrever o tom principal com base em marcadores linguísticos."""
    user_df = df[df['user'] == user]
    caps = user_df['caps_ratio'].mean()
    excl = user_df['exclamation_count'].mean()
    quest = user_df['question_count'].mean()
    parts = []
    if caps > 0.3:
        parts.append('Enfático (alto uso de maiúsc.)')
    elif caps > 0.15:
        parts.append('Moderadamente enfático')
    else:
        parts.append('Uso padrão de caixa')
    if excl > 1.0:
        parts.append('exclamatório')
    if quest > 0.5:
        parts.append('interrogativo')
    return ', '.join(parts)

def get_media_preference(user):
    """Descrever o padrão de uso de mídia."""
    user_df = df[df['user'] == user]
    media_dist = user_df['media_type'].value_counts(normalize=True)
    top_type = media_dist.index[0] if len(media_dist) > 0 else 'none'
    top_pct = media_dist.values[0] * 100 if len(media_dist) > 0 else 0
    return f'{top_type.title()} dominante ({top_pct:.0f}%)'

def get_engagement_style(user):
    """Descrever a abordagem de engajamento com o público."""
    user_df = df[df['user'] == user]
    mention_rate = user_df['mention_count'].mean()
    reply_pct = user_df['is_reply'].mean() * 100
    quote_pct = user_df['is_quote'].mean() * 100
    if reply_pct > 20:
        style = 'Conversacional'
    elif quote_pct > 15:
        style = 'Orientado a comentários'
    else:
        style = 'Orientado a transmissão'
    return f'{style} (respostas: {reply_pct:.0f}%, citações: {quote_pct:.0f}%, menções/tweet: {mention_rate:.1f})'

# Construir a tabela comparativa
comparison = {
    'Tom Principal': {
        'Trump': get_tone_descriptor('trump'),
        'AOC': get_tone_descriptor('aoc')
    },
    'Foco de Conteúdo': {
        'Trump': get_top_topic('trump'),
        'AOC': get_top_topic('aoc')
    },
    'Tam. Médio do Texto': {
        'Trump': f"{df[df['user']=='trump']['text_length'].mean():.0f} caract. / {df[df['user']=='trump']['word_count'].mean():.0f} palavras",
        'AOC': f"{df[df['user']=='aoc']['text_length'].mean():.0f} caract. / {df[df['user']=='aoc']['word_count'].mean():.0f} palavras"
    },
    'Estratégia de Hashtags': {
        'Trump': get_hashtag_strategy('trump'),
        'AOC': get_hashtag_strategy('aoc')
    },
    'Preferência de Mídia': {
        'Trump': get_media_preference('trump'),
        'AOC': get_media_preference('aoc')
    },
    'Compartilhamento de URLs': {
        'Trump': f"{(df[df['user']=='trump']['url_count']>0).mean()*100:.0f}% dos tweets contêm URLs",
        'AOC': f"{(df[df['user']=='aoc']['url_count']>0).mean()*100:.0f}% dos tweets contêm URLs"
    },
    'Uso de Emoji': {
        'Trump': f"{df[df['user']=='trump']['has_emoji'].mean()*100:.0f}% dos tweets",
        'AOC': f"{df[df['user']=='aoc']['has_emoji'].mean()*100:.0f}% dos tweets"
    },
    'Estilo de Engajamento': {
        'Trump': get_engagement_style('trump'),
        'AOC': get_engagement_style('aoc')
    },
}

comparison_df = pd.DataFrame(comparison).T
comparison_df.index.name = 'Dimensão'

# Exibir tabela estilizada
styled = comparison_df.style.set_caption(
    'Comparação de Identidade Digital: Trump vs AOC'
).set_table_styles([
    {'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold'), ('margin-bottom', '10px')]},
    {'selector': 'th', 'props': [('background-color', '#f0f0f0'), ('font-weight', 'bold'), ('text-align', 'center'), ('padding', '8px')]},
    {'selector': 'td', 'props': [('text-align', 'left'), ('padding', '8px'), ('max-width', '300px')]},
    {'selector': 'th.row_heading', 'props': [('text-align', 'left'), ('font-weight', 'bold')]},
]).set_properties(
    subset=['Trump'], **{'background-color': '#fce4e4'}
).set_properties(
    subset=['AOC'], **{'background-color': '#e4ecf4'}
)

display(styled)

# Também imprimir texto simples
print("\nVersão em texto simples:")
print("=" * 90)
print(f"{'Dimensão':<22} {'Trump':<34} {'AOC':<34}")
print("=" * 90)
for dim, values in comparison.items():
    trump_val = values['Trump'][:32]
    aoc_val = values['AOC'][:32]
    print(f"{dim:<22} {trump_val:<34} {aoc_val:<34}")

## 8. Principais Conclusões

**Características do Texto:**
- As diferenças no tamanho do texto, contagem de palavras e verbosidade entre os dois usuários revelam abordagens retóricas contrastantes
- Padrões de capitalização sugerem diferentes níveis de intensidade retórica
- O uso de pontuação (exclamações/interrogações) e padrões de emoji completam o quadro estilístico

**Vocabulário e Escolha de Palavras:**
- As palavras mais frequentes revelam as prioridades temáticas de cada político
- O vocabulário distintivo -- palavras únicas no léxico de cada usuário -- reforça diferenças de identidade digital

**Estratégias de Hashtags:**
- A frequência e estilo de uso de hashtags diferem significativamente
- As hashtags servem como branding, mobilização ou marcadores temáticos
- A sobreposição ou divergência de hashtags entre os usuários reflete mundos políticos distintos

**Fontes de Informação (URLs e Links):**
- A frequência de compartilhamento de links externos revela estratégias de criação vs. curadoria de conteúdo

**Redes de Menções:**
- Quem cada usuário referencia revela sua rede política e prioridades de engajamento
- Padrões de menção cruzada mostram se eles se engajam diretamente um com o outro

**Ênfase em Tópicos:**
- Quais tópicos dominam o feed de cada usuário reflete suas agendas e prioridades
- A sobreposição e divergência de tópicos funcionam como indicadores de priorização de pautas

**Perfis de Identidade Digital:**
- Trump e AOC constroem identidades digitais distintas por meio de suas escolhas de conteúdo
- Essas abordagens contrastantes refletem filosofias diferentes sobre comunicação política online

---

*Nota metodológica: A classificação de tópicos baseada em palavras-chave e a análise de frequência de palavras fornecem uma estrutura quantitativa para a análise de discurso, mas devem ser complementadas pela leitura qualitativa aprofundada de tweets individuais.*

### Próximos Passos
- **Notebook 4**: O que as pessoas dizem? Análise de respostas da comunidade
- **Notebook 5**: O poder das imagens -- análise visual comparativa